# 06: メンテナンス（Compaction）と GCP / Airflow 実践リファレンス

## このノートブックで学習できること

- 少量のデータを頻繁に `append` することで発生する Small File Problem の再現と、Compaction（ファイル結合）によるファイル数削減・統計情報整理の効果（Iceberg / Delta Lake 両方）
- ローカルで学んだ内容を GCP（東京リージョン `asia-northeast1`）の BigQuery（BigLake Managed Tables for Apache Iceberg）× Databricks × Apache Airflow 2 系へ適用するための実践設計とコードテンプレート

## 到達ゴール

- Small File Problem が発生する原因と Compaction による改善効果を実測値で説明できる
- BigLake Iceberg の DDL、クロスプロジェクト取り込み、Airflow 2 系 DAG 移行、GCS インフラ・IAM 3 層防御、および将来的な Databricks（Delta UniForm）移行ロードマップの設計指針を把握する


## 1. Small File Problem の再現と Compaction

このセルでは、50 回の小さな `append` を行って意図的に極小 Parquet ファイルを 50 個生成し、Iceberg の再書き込みによる Compaction と Delta Lake の `optimize.compact()` を実行してファイル数の変化を確認します。


In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.catalog_helper import (
    LANDING_DIR,
    LAKEHOUSE_DIR,
    get_duckdb_conn,
    get_iceberg_catalog,
    init_storage,
    register_iceberg_table,
)
import time
import polars as pl
from deltalake import DeltaTable, write_deltalake

catalog = get_iceberg_catalog()
if catalog.table_exists("bronze.small_files_demo"):
    catalog.drop_table("bronze.small_files_demo")

sample_src = pl.read_csv(LANDING_DIR / "day1" / "crm_customers.csv").select(["customer_id", "membership_tier", "status"])

# Iceberg で 50 回細切れ追記
small_tbl = catalog.create_table("bronze.small_files_demo", schema=sample_src.slice(0, 1).to_arrow().schema)
for i in range(50):
    small_tbl.append(sample_src.slice(i * 10, 10).to_arrow())

before_files = len(list(small_tbl.scan().plan_files()))

# Iceberg Compaction（読み出して1ファイルに再構成）
compacted_arrow = small_tbl.scan().to_arrow()
small_tbl.overwrite(compacted_arrow)
after_files = len(list(small_tbl.scan().plan_files()))

# Delta Lake でも同様に 50 回追記して optimize.compact()
delta_small_path = LAKEHOUSE_DIR / "bronze" / "small_files_delta"
for i in range(50):
    write_deltalake(str(delta_small_path), sample_src.slice(i * 10, 10).to_arrow(), mode="overwrite" if i == 0 else "append")

dt_small = DeltaTable(str(delta_small_path))
delta_before = len(dt_small.file_uris())
dt_small.optimize.compact()
delta_after = len(DeltaTable(str(delta_small_path)).file_uris())

print(f"Iceberg 有効 Parquet 数: {before_files} -> Compaction 後: {after_files}")
print(f"Delta   有効 Parquet 数: {delta_before} -> optimize.compact() 後: {delta_after}")


Iceberg・Delta Lake ともに 50 個あった小さな Parquet ファイルが Compaction によって `1` 個に統合され、メタデータの肥大化とファイルオープンのオーバーヘッドが解消されました。

---

## 2. GCP 本番運用の設計リファレンス（`asia-northeast1`）

以下は、ローカルで検証した OTF の設計思想を GCP 上の **BigQuery（BigLake Managed Tables for Apache Iceberg）× Databricks × Apache Airflow 2 系** に適用する際の実践テンプレート集です。

### 2.1 BigLake Iceberg DDL とスキャン最適化

BigQuery では `CLUSTER BY` によってデータがソート・配置され、Iceberg の Manifest File 統計情報と連動した 3 段階プルーニングが機能します。また、日付列などに対する Hidden Partitioning 相当の最適化により、利用者がパーティション列を意識せずにフィルタを書けます。

```sql
CREATE TABLE `my-lakehouse-prod.silver.fct_sales` (
  sale_id STRING NOT NULL,
  date_key INT64 NOT NULL,
  customer_key INT64 NOT NULL,
  product_key INT64 NOT NULL,
  store_key INT64 NOT NULL,
  payment_method STRING,
  quantity INT64,
  gross_amount INT64,
  discount_amount INT64,
  net_amount INT64,
  purchased_at TIMESTAMP
)
CLUSTER BY date_key, store_key
WITH CONNECTION `my-lakehouse-prod.asia-northeast1.biglake-conn`
OPTIONS (
  file_format = 'PARQUET',
  table_format = 'ICEBERG',
  storage_uri = 'gs://lakehouse-main-asia-ne1/silver/fct_sales'
);
```

### 2.2 クロスプロジェクト取り込み（SQL / dbt）

別 GCP プロジェクト（`source-dwh-prod`）の BigQuery ネイティブテーブルから、自プロジェクトの GCS 上の BigLake Iceberg テーブルへ直接書き込む設定例です。

- **SQL の場合:**

```sql
MERGE `my-lakehouse-prod.silver.crm_customers` T
USING (
  SELECT customer_id, membership_tier, status, updated_at
  FROM `source-dwh-prod.crm.customers_daily`
  WHERE DATE(updated_at) = CURRENT_DATE('Asia/Tokyo')
) S
ON T.customer_id = S.customer_id
WHEN MATCHED THEN
  UPDATE SET membership_tier = S.membership_tier, status = S.status, updated_at = S.updated_at
WHEN NOT MATCHED THEN
  INSERT (customer_id, membership_tier, status, updated_at)
  VALUES (S.customer_id, S.membership_tier, S.status, S.updated_at);
```

- **dbt（`dbt-bigquery`）モデル設定例:**

```sql
{{ config(
    materialized='incremental',
    unique_key='customer_id',
    table_format='iceberg',
    cluster_by=['membership_tier'],
    partition_by=None
) }}
SELECT customer_id, membership_tier, status, updated_at
FROM {{ source('source_dwh', 'customers_daily') }}
```

### 2.3 Airflow 2 系パイプライン移行パターン

従来の `GCSToBigQueryOperator`（Landing ファイルをそのままネイティブテーブルへロード）から、`BigQueryInsertJobOperator` を用いて「Landing の一時外部テーブル定義 ＋ `SAFE_CAST` / `_rescued_data` 退避 ＋ Iceberg テーブルへの `MERGE`」へ置き換える DAG テンプレートです。

```python
from datetime import datetime
from airflow import DAG
from airflow.providers.google.cloud.operators.bigquery import BigQueryInsertJobOperator

MERGE_SQL = """
CREATE OR REPLACE TEMP EXTERNAL TABLE tmp_pos_landing
OPTIONS (
  format = 'NEWLINE_DELIMITED_JSON',
  uris = ['gs://data-landing-asia-ne1/pos/{{ ds }}/*.jsonl']
);

MERGE `my-lakehouse-prod.bronze.pos_transactions` T
USING (
  SELECT
    transaction_id,
    store_id,
    customer_id,
    product_id,
    SAFE_CAST(quantity AS INT64) AS quantity,
    CASE
      WHEN SAFE_CAST(quantity AS INT64) IS NULL THEN TO_JSON_STRING(STRUCT(quantity))
      ELSE NULL
    END AS _rescued_data,
    SAFE_CAST(total_amount AS INT64) AS total_amount,
    TIMESTAMP(purchased_at) AS purchased_at
  FROM tmp_pos_landing
) S
ON T.transaction_id = S.transaction_id
WHEN NOT MATCHED THEN
  INSERT ROW;
"""

with DAG(
    dag_id="ingest_pos_to_iceberg",
    start_date=datetime(2026, 11, 1),
    schedule="@daily",
    catchup=False,
) as dag:
    ingest_task = BigQueryInsertJobOperator(
        task_id="merge_pos_iceberg",
        location="asia-northeast1",
        configuration={"query": {"query": MERGE_SQL, "useLegacySql": False}},
    )
```

### 2.4 GCS インフラ・IAM・バケット保護と移行ロードマップ

1. **2 バケット構成とライフサイクル（`asia-northeast1`）**
   - Landing バケット（`data-landing-asia-ne1`）: Standard クラス＋Object Lifecycle Management（OLM）で 30〜90 日後に自動削除。
   - Lakehouse Main バケット（`lakehouse-main-asia-ne1`）: Standard クラス（または Autoclass）。Iceberg / Delta のタイムトラベルや参照中ファイルが消えてテーブルが破損するのを防ぐため、OLM による自動削除は設定しません。

2. **IAM 最小権限設計**
   - 人間のユーザー（アナリスト・開発者含む）には GCS バケットの直接権限（`roles/storage.objectViewer` 等）を一切付与せず、BigLake 接続のサービスアカウントおよび Databricks Unity Catalog 用サービスアカウントに限定します。データ参照はすべてカタログの権限（BigQuery / Unity Catalog）経由に統一します。

3. **単一バケット集約リスクに対する 3 層防御**
   - 第 1 層（IaC 防御）: Terraform の `lifecycle { prevent_destroy = true }` で誤削除をブロック。
   - 第 2 層（物理削除猶予）: GCS Soft Delete（例: 保持期間 7〜30 日）を有効化し、万一のバケット・オブジェクト削除から復元可能にする。
   - 第 3 層（特権オーバーライド対策）: データ基盤専用の独立した GCP プロジェクトへ分離したうえで、**IAM 拒否ポリシー（IAM Deny Policy）** を適用し、プロジェクトの Owner / Editor であっても許可されたサービスアカウント・break-glass 用アカウント以外からの `storage.buckets.delete` や `storage.objects.delete` を拒否します。

4. **BigQuery（Iceberg）→ Databricks（Delta Lake）移行 3 フェーズロードマップ**
   - **Phase 1（BigQuery 主軸）**: BigQuery BigLake Managed Tables（Iceberg）で Bronze / Silver / Gold を運用し、外部エンジンからは Iceberg メタデータ経由で参照する。
   - **Phase 2（共存・相互運用）**: Databricks 側の Unity Catalog に GCS 上のテーブルを登録し、Delta Lake **UniForm**（Delta コミット時に Iceberg メタデータを同時生成）を有効化して BigQuery からの読み取り互換性を維持する。
   - **Phase 3（Databricks 主軸への切り替え）**: 既存 Iceberg テーブルに対して `CONVERT TO DELTA`（または UniForm 有効の Delta テーブルへ段階切替）を実施し、書き込み・ガバナンス（Unity Catalog）を Databricks へ集約する。
